# 朴素贝叶斯文本分类

**面试回答：**多项式朴素贝叶斯以类别先验加词条件概率比较后验；条件独立是假设，Laplace 平滑和对数计算是必要的数值工程。

## 真实案例

8 条客服短工单被分为退款和物流，词袋计数来自离线分词，目标是给新工单路由。

In [1]:
import numpy as np  # 导入 NumPy 手写朴素贝叶斯。
words=np.array(['退款','发票','物流','延迟'])  # 定义可解释的词表。
ticket=np.array(['S01','S02','S03','S04','S05','S06','V01','V02'])  # 构造工单编号。
x=np.array([[2,1,0,0],[1,0,0,0],[1,1,0,0],[0,1,0,0],[0,0,2,1],[0,0,1,2],[1,0,1,0],[0,1,0,2]],dtype=float)  # 构造词频特征。
y=np.array([0,0,0,0,1,1,0,1])  # 标记退款零类与物流一类。
print('工单 | 词袋 [退款 发票 物流 延迟] | 类别')  # 输出工单数据表头。
for n,row,label in zip(ticket,x,y):  # 逐条展示语义样本。
    print(n,row.astype(int).tolist(),label)  # 输出一条词袋记录。

工单 | 词袋 [退款 发票 物流 延迟] | 类别
S01 [2, 1, 0, 0] 0
S02 [1, 0, 0, 0] 0
S03 [1, 1, 0, 0] 0
S04 [0, 1, 0, 0] 0
S05 [0, 0, 2, 1] 1
S06 [0, 0, 1, 2] 1
V01 [1, 0, 1, 0] 0
V02 [0, 1, 0, 2] 1


## Baseline / 基线

基线总是输出训练集中更多的退款类，完全不读取词。

In [2]:
train=np.arange(6)  # 定义前六条为训练工单。
valid=np.arange(6,8)  # 定义后两条为验证工单。
majority=int(np.bincount(y[train]).argmax())  # 找到训练集中多数类别。
baseline=np.full(len(valid),majority)  # 对所有验证工单输出多数类。
baseline_acc=float(np.mean(baseline==y[valid]))  # 计算多数类基线准确率。
print('多数类基线:',majority,'准确率=',baseline_acc)  # 输出基线结果。

多数类基线: 0 准确率= 0.5


In [3]:
class_count=np.bincount(y[train],minlength=2)  # 统计训练集每类工单数。
prior=(class_count+1)/(len(train)+2)  # 用平滑计算类别先验。
word_count=np.vstack([x[train][y[train]==c].sum(axis=0) for c in range(2)])  # 汇总每类词频。
likelihood=(word_count+1)/(word_count.sum(axis=1,keepdims=True)+len(words))  # 用 Laplace 平滑计算条件概率。
log_score=np.log(prior)+x[valid]@np.log(likelihood).T  # 在对数域累加后验分数。
pred=log_score.argmax(axis=1)  # 选择最大后验类别。
acc=float(np.mean(pred==y[valid]))  # 计算朴素贝叶斯验证准确率。
print('先验:',np.round(prior,3),'词条件概率:',np.round(likelihood,3))  # 输出模型中间量。
print('验证对数后验:',np.round(log_score,2),'预测:',pred.tolist())  # 输出预测依据。

先验: [0.625 0.375] 词条件概率: [[0.455 0.364 0.091 0.091]
 [0.1   0.1   0.4   0.4  ]]
验证对数后验: [[-3.66 -4.2 ]
 [-6.28 -5.12]] 预测: [0, 1]


## 结果解读

词独立并不要求真实语言独立；它是用于快速估计后验的近似。对数后验无需再归一化即可比较类别。

In [4]:
print('模型 | 验证准确率 | 解释')  # 输出结果表头。
print(f'多数类 | {baseline_acc:.2f} | 不读取文本')  # 输出基线行。
print(f'朴素贝叶斯 | {acc:.2f} | 使用平滑词概率')  # 输出主要模型行。
print('生产差距：需版本化分词/词表、处理 OOV、按会话切分并监控类别漂移。')  # 说明生产边界。

模型 | 验证准确率 | 解释
多数类 | 0.50 | 不读取文本
朴素贝叶斯 | 1.00 | 使用平滑词概率
生产差距：需版本化分词/词表、处理 OOV、按会话切分并监控类别漂移。


## 失败案例与修复

不做平滑时，未在某类出现的词会使该类整条工单概率为零；修复是加一平滑。

In [5]:
raw_likelihood=word_count/word_count.sum(axis=1,keepdims=True)  # 故意计算未平滑条件概率。
with np.errstate(divide='ignore', invalid='ignore'):  # 静默捕获教学反例中的 log(0) 数值警告。
    raw_score=np.log(prior)+x[valid]@np.log(raw_likelihood).T  # 计算包含 log(0) 的失败分数。
print('失败：未平滑是否含无穷=',bool(np.isinf(raw_score).any()))  # 输出零概率导致的问题。
print('修复：平滑后是否有限=',bool(np.isfinite(log_score).all()))  # 输出数值修复结果。
print('注意：本实验数据很小，不能把准确率外推为真实客服效果。')  # 明确教学边界。

失败：未平滑是否含无穷= False
修复：平滑后是否有限= True
注意：本实验数据很小，不能把准确率外推为真实客服效果。


In [6]:
assert len(ticket)>=5  # 保护工单数量。
assert acc>=baseline_acc  # 保护文本模型不弱于多数类基线。
assert np.allclose(likelihood.sum(axis=1),1)  # 保护条件概率归一化。
assert np.isfinite(log_score).all()  # 保护平滑后数值有限。